In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
filename = "input.txt"

urllib.request.urlretrieve(url, filename)
print(f"✅ 下载完成: {filename}")


In [28]:
with open("input.txt","r",encoding="utf-8") as f:
    text = f.read()
    

In [31]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [32]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
vocab_size



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


65

In [33]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i:ch for i , ch in enumerate(chars)}
encode = lambda s:[stoi[c] for c in s]
decode = lambda l : ''.join([itos[i] for i in l])

print(encode("hii there"))
print(decode(encode("hii there")))
                            

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [34]:
import torch
data = torch.tensor(encode(text),dtype= torch.long)
print(data.shape,data.dtype)
print(data[:1000])


torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [35]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]


In [36]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [37]:
x = train_data[:block_size]
y = train_data[1:block_size +1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [38]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size]for i in ix])
    y = torch.stack([data[i+1:i+1+block_size] for i in ix])
    return x,y

xb, yb = get_batch("train")
print("input: ")
print(xb.shape)
print(xb)
print("traget: ")
print(yb.shape)
print(yb)

input: 
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
traget: 
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [39]:
print(xb)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [40]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanaguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

    def forward(self, idx, targets = None):
        logits = self.token_embedding_table(idx)
    
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)
            
        return logits, loss

    def generate(self,idx,max_new_tokens):
        for _ in range(max_new_tokens): 
            logits, loss = self(idx)
            
            logits = logits[:, -1, :] 
            
            probs = F.softmax(logits, dim=-1)
            
            idx_next = torch.multinomial(probs, num_samples=1)
            
            idx = torch.cat((idx, idx_next), dim=1)
            
        return idx

m = BigramLanaguageModel(vocab_size)
out,loss= m(xb,yb)
print(out.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1,1),dtype= torch.long),max_new_tokens = 100)[0].tolist()))


torch.Size([4, 8, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

Sr?qP-QWktXoL&jLDJgOLVz'RIoDqHdhsV&vLLxatjscMpwLERSPyao.qfzs$Ys$zF-w,;eEkzxjgCKFChs!iWW.ObzDnxA Ms$3


In [41]:
#预训练
optmizer = torch.optim.AdamW(m.parameters(), lr= 1e-3)

In [42]:
batch_size = 32;
for steps in range(10000):
    xb,yb = get_batch('train')

    logits, loss = m(xb,yb)
    optmizer.zero_grad(set_to_none=True)
    loss.backward()
    optmizer.step()

    print(loss.item())

4.704006195068359
4.721118927001953
4.653193473815918
4.706261157989502
4.780904293060303
4.751267910003662
4.8395490646362305
4.667973041534424
4.743716716766357
4.774043083190918
4.6908278465271
4.789143085479736
4.61777925491333
4.650947093963623
4.886447429656982
4.703796863555908
4.757591724395752
4.65510892868042
4.709283828735352
4.6745147705078125
4.760501384735107
4.7892632484436035
4.653748512268066
4.6619181632995605
4.673007488250732
4.66577672958374
4.7301106452941895
4.755304336547852
4.712186813354492
4.745501518249512
4.726755619049072
4.735108375549316
4.777461051940918
4.643350601196289
4.6651835441589355
4.79764461517334
4.717412948608398
4.683647155761719
4.81886100769043
4.613771915435791
4.573785781860352
4.560741901397705
4.81563138961792
4.6061553955078125
4.619696140289307
4.725419521331787
4.650487899780273
4.5941481590271
4.7202863693237305
4.699342250823975
4.6724138259887695
4.727972984313965
4.66152286529541
4.616766929626465
4.599857807159424
4.6533403396

In [43]:
print(decode(m.generate(idx = torch.zeros((1,1),dtype= torch.long),max_new_tokens = 1000)[0].tolist()))


Iyoteng h hasbe pave pirance
Rie hicomyonthar's
Plinseard ith henoure wounonthioneir thondy, y heltieiengerofo'dsssit ey
KIN d pe wither vouprrouthercc.
hathe; d!
My hind tt hinig t ouchos tes; st yo hind wotte grotonear 'so it t jod weancotha:
h hay.JUCle n prids, r loncave w hollular s O:
HIs; ht anjx?

DUThinqunt.

LaZAnde.
athave l.
KEONH:
ARThanco be y,-hedarwnoddy scace, tridesar, wnl'shenous s ls, theresseys
PlorseelapinghiybHen yof GLUCEN t l-t E:
I hisgothers je are!-e!
QLYotouciullle'z,
Thitertho s?
NDan'spererfo cist ripl chys er orlese;
Yo jehof h hecere ek? wferommot mowo soaf yoit, ince his, t, f at. fal whetrimy bupof tor atha Bu!
JOutho f cimimave.
NEDUSt cir selle p wie wede
Ro n apenor f'Y tover witys an sh d w t e w!
CEOntiretoaveE IINpe, theck. cung.
ORIsthies hacin benqurd bll, d a r w wistatsowor ath
Fivet bloll ang a-I theeancu,
LINCI'T:
Sarry t I Ane sze t
LCKI thit,
n.
Faure ds ppplirn!
meftou ow pring, avewist th;
TENTEMETCI gienco, An he waro whiougou he s i

In [44]:
#self-attention


torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [45]:
xbow = torch.zeros(B,T,C)
for b in range(B):
    for t in range(T):
        xpre = x[b,:t+1]
        xbow[b,t] = torch.mean(xpre,0)

In [46]:
torch.manual_seed(1337)
B,T,C = 4,8,32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C,head_size,bias =False)
query = nn.Linear(C,head_size, bias = False)
value = nn.Linear(C,head_size, bias =False)

k = key(x)
q = query(x)

wei = q @ k.transpose(-2,-1)
tril = torch.tril(torch.ones(T,T))

wei = wei.masked_fill(tril ==0, float('-inf'))
wei = torch.softmax(wei,dim = -1)

v = value(x)
out = wei@v

print(out)
out.shape

tensor([[[-1.5713e-01,  8.8009e-01,  1.6152e-01, -7.8239e-01, -1.4289e-01,
           7.4676e-01,  1.0068e-01, -5.2395e-01, -8.8726e-01,  1.9068e-01,
           1.7616e-01, -5.9426e-01, -4.8124e-01, -4.8598e-01,  2.8623e-01,
           5.7099e-01],
         [ 6.7643e-01, -5.4770e-01, -2.4780e-01,  3.1430e-01, -1.2799e-01,
          -2.9521e-01, -4.2962e-01, -1.0891e-01, -4.9282e-02,  7.2679e-01,
           7.1296e-01, -1.1639e-01,  3.2665e-01,  3.4315e-01, -7.0975e-02,
           1.2716e+00],
         [ 4.8227e-01, -1.0688e-01, -4.0555e-01,  1.7696e-01,  1.5811e-01,
          -1.6967e-01,  1.6217e-02,  2.1509e-02, -2.4903e-01, -3.7725e-01,
           2.7867e-01,  1.6295e-01, -2.8951e-01, -6.7610e-02, -1.4162e-01,
           1.2194e+00],
         [ 1.9708e-01,  2.8561e-01, -1.3028e-01, -2.6552e-01,  6.6781e-02,
           1.9535e-01,  2.8073e-02, -2.4511e-01, -4.6466e-01,  6.9287e-02,
           1.5284e-01, -2.0324e-01, -2.4789e-01, -1.6213e-01,  1.9474e-01,
           7.6778e-01],
    

torch.Size([4, 8, 16])

In [47]:
import torch
import torch.nn as nn

# 词表 4 个词，每个词映射为 5 维向量（故意让 C≠vocab_size 方便观察）
emb = nn.Embedding(num_embeddings=4, embedding_dim=5)

# 输入：2 条句子，每句 3 个 token
idx = torch.tensor([[0, 2, 1],
                    [3, 0, 2]])  # 形状 (B=2, T=3)

out = emb(idx)
print(out)
B,T,C = out.shape
out = out.view(B*T,C)
print("输入形状:", idx.shape)   # torch.Size([2, 3])
print(idx)
print("输出形状:", out.shape)   # torch.Size([2, 3, 5]) → (B, T, C)
print(out)

print(emb)

tensor([[[-0.0399,  1.0038, -1.1435,  1.8040,  1.5473],
         [ 1.7253,  2.2931,  1.1496,  1.0411,  0.0580],
         [-0.6886, -0.4414,  1.2790, -0.7990, -1.9086]],

        [[-1.6868, -1.4225,  0.0888, -0.8783,  1.1719],
         [-0.0399,  1.0038, -1.1435,  1.8040,  1.5473],
         [ 1.7253,  2.2931,  1.1496,  1.0411,  0.0580]]],
       grad_fn=<EmbeddingBackward0>)
输入形状: torch.Size([2, 3])
tensor([[0, 2, 1],
        [3, 0, 2]])
输出形状: torch.Size([6, 5])
tensor([[-0.0399,  1.0038, -1.1435,  1.8040,  1.5473],
        [ 1.7253,  2.2931,  1.1496,  1.0411,  0.0580],
        [-0.6886, -0.4414,  1.2790, -0.7990, -1.9086],
        [-1.6868, -1.4225,  0.0888, -0.8783,  1.1719],
        [-0.0399,  1.0038, -1.1435,  1.8040,  1.5473],
        [ 1.7253,  2.2931,  1.1496,  1.0411,  0.0580]],
       grad_fn=<ViewBackward0>)
Embedding(4, 5)


In [50]:
torch.manual_seed(1337)
def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size]for i in ix])
    y = torch.stack([data[i+1:i+1+block_size] for i in ix])
    return x,y

In [52]:
torch.manual_seed(1337)
m = BigramLanguageModel().to(device)
optimizer = torch.optim.AdamW(m.parameters(), lr=3e-4)

for step in range(5000):
    xb, yb = get_batch('train')
    xb, yb = xb.to(device), yb.to(device) 
    logits, loss = m(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    if step % 500 == 0:
        print(f"step {step:5d} | loss: {loss.item():.4f}")

step     0 | loss: 4.3019
step   500 | loss: 1.8948
step  1000 | loss: 1.6121
step  1500 | loss: 1.4967
step  2000 | loss: 1.4159
step  2500 | loss: 1.3681
step  3000 | loss: 1.3163
step  3500 | loss: 1.2829
step  4000 | loss: 1.2666
step  4500 | loss: 1.2384


In [51]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# 超参数 
vocab_size = 65
batch_size = 64;
n_embd = 384
block_size = 256
n_head = 6
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4* n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(0.2),
        )

    def forward(self,x):
        return self.net(x)
        

class Head(nn.Module):
    def __init__(self, head_size): 
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd,n_embd)
        
    def forward(self, x):
        out =  torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out


class Block(nn.Module):
    def __init__(self,n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)

    def forward(self,x):
        x = x + self.sa(x)
        x = x + self.ffwd(x)
        return x


class BigramLanguageModel(nn.Module):  # ✅ 修复类名拼写
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            Block(n_embd, n_head = 4),
            Block(n_embd, n_head = 4),
            Block(n_embd, n_head = 4),
            nn.LayerNorm(n_embd),
        )
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_embd = self.token_embedding_table(idx)  # (B, T, n_embd)
        pos_embd = self.position_embedding_table(torch.arange(T, device=idx.device))  # (T, n_embd)
        
        x = token_embd + pos_embd  # (B, T, n_embd)
        x = self.blocks(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)
            
        return logits, loss

    @torch.no_grad() 
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [55]:
print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long,device=device), max_new_tokens=1000)[0].tolist()))


And trust things to end, I' follow enter.
There little pery, to his house of his with three
Scowling and reigns; take the where he absole kon
the envidestory, here's doom and shall you
heart a world wrongfully was them.

EXTPA:
Go thou art my lords, pray, give that enough secret
so appointing to thy praydress good, I know
bound of that, and I belike a grave done
to Richard' us: dismiss mains my fresh
To perfight again, trusts upen to the resolvion,
of no glorious doing, our tragery, but I
fraunt, let thy leaving no advise services, and
fall committed thee, for my revil
Whereto the fine is words: is let justice,
a ministerity ant unto to dy death, a fine
'Gain far doth loss wially peace!'

PERDITA:
Agaim on, little prepared by my hand,
And but the king; and something to coterps:
I for I love, and let them trust wir; and,
and now in my eyes a Verian new work!
Deperdisdain--for these these name made or my primes.

PHEN ELIZABETH:
Lord High of the princes hath beet me
Thine ears him be lo

In [ ]:
print(torch.cuda.is_available())